In [1]:
import requests
import time
import random
import hashlib

In [2]:
session = requests.Session()

headers = {
    "User-Agent": "Mozilla/5.0",
    "Accept-Language": "en-US,en;q=0.9",
    "Accept-Encoding": "gzip, deflate, br",
}

def fetch_nse(symbol):
    url = f"https://www.nseindia.com/api/quote-equity?symbol={symbol}"
    try:
        response = session.get(url, headers=headers, timeout=5)
        if response.status_code != 200:
            return None
        return response.json()
    except:
        return None

In [3]:
def extract_features(data):
    features = {}

    price = data.get('priceInfo', {})
    security = data.get('securityInfo', {})

    features['price'] = price.get('lastPrice')
    features['pChange'] = price.get('pChange')
    features['volume'] = data.get('preOpenMarket', {}).get('totalTradedVolume')

    # additional features
    features['company'] = security.get('companyName')
    features['industry'] = security.get('industry')
    features['symbol'] = security.get('symbol')

    return features

In [4]:
symbols = [
    "RELIANCE","TCS","INFY","HDFCBANK","ICICIBANK",
    "SBIN","LT","ITC","HINDUNILVR","KOTAKBANK",
    "AXISBANK","ASIANPAINT","MARUTI","SUNPHARMA",
    "TITAN","ULTRACEMCO","BAJFINANCE","WIPRO",
    "NTPC","POWERGRID"
]

In [5]:
price_history = {}
volume_history = {}

k = 5
mg_counter = {}

max_zeros = 0

b = 4
m = 2 ** b
registers = [0] * m

In [6]:
def update_mg(item):
    global mg_counter

    if item in mg_counter:
        mg_counter[item] += 1

    elif len(mg_counter) < k:
        mg_counter[item] = 1

    else:
        remove_keys = []
        for key in mg_counter:
            mg_counter[key] -= 1
            if mg_counter[key] == 0:
                remove_keys.append(key)

        for key in remove_keys:
            del mg_counter[key]

In [7]:
def hash_value(item):
    return int(hashlib.md5(item.encode()).hexdigest(), 16)

def trailing_zeros(x):
    if x == 0:
        return 0
    return (x & -x).bit_length() - 1

def update_fm(item):
    global max_zeros

    if item is None:   # ✅ FIX
        return

    h = hash_value(item)
    tz = trailing_zeros(h)
    max_zeros = max(max_zeros, tz)

def estimate_distinct():
    return 2 ** max_zeros

In [8]:
def update_hll(item):
    global registers

    if item is None:   # ✅ FIX
        return

    h = hash_value(item)

    bucket = h & (m - 1)
    w = h >> b

    tz = trailing_zeros(w)

    registers[bucket] = max(registers[bucket], tz)

def estimate_hll():
    Z = sum([2 ** (-r) for r in registers])
    E = (0.7213 / (1 + 1.079 / m)) * (m ** 2) / Z
    return int(E)

In [9]:
for _ in range(50):   # runs 50 iterations

    sym = random.choice(symbols)
    print("\nTrying:", sym)   # 🔍 debug

    data = fetch_nse(sym)

    # 🚨 If API fails, use fallback data (so output always comes)
    if data is None:
        print("⚠️ API Failed → Using Dummy Data")
        data = {
            "priceInfo": {"lastPrice": random.uniform(100, 3000), "pChange": 0},
            "securityInfo": {"symbol": sym},
            "preOpenMarket": {"totalTradedVolume": random.randint(1000, 10000)}
        }

    features = extract_features(data)

    sym = features.get('symbol')
    price = features.get('price')
    volume = features.get('volume')

    # ✅ skip bad data (extra safety)
    if sym is None or price is None or volume is None:
        print("❌ Skipping bad data")
        continue

    # -------- STORE PRICE --------
    if sym not in price_history:
        price_history[sym] = []
    price_history[sym].append(price)

    # -------- STORE VOLUME --------
    if sym not in volume_history:
        volume_history[sym] = []
    volume_history[sym].append(volume)

    # -------- UPDATE ALGORITHMS --------
    update_mg(sym)
    update_fm(sym)
    update_hll(sym)

    print("Stock:", sym)
    print("Price:", price, "| Volume:", volume)
    print("Top Stocks:", mg_counter)

    # -------- RETURNS --------
    for stock in mg_counter:
        if stock in price_history and len(price_history[stock]) > 1:
            prices = price_history[stock]
            ret = (prices[-1] - prices[-2]) / prices[-2]
            print(f"Return ({stock}):", round(ret, 5))

    print("Distinct (FM):", estimate_distinct())
    print("Distinct (HLL):", estimate_hll())
    print("-----------")

    time.sleep(0.5)